#  Notebook 4: Duplicate Data Handling

Simple examples using Pandas. Each topic includes Definition, Business Example, AI/ML Use, Python Example, and Explanation.

In [1]:
import pandas as pd

df = pd.DataFrame({
    "Customer_ID": [101, 102, 102, 103, 104, 104],
    "Name": ["Asha", "Ravi", "Ravi", "Meena", "John", "John"],
    "Order_ID": [501, 502, 502, 503, 504, 505],
    "Amount": [500, 800, 800, 600, 700, 700]
})
print(df)

   Customer_ID   Name  Order_ID  Amount
0          101   Asha       501     500
1          102   Ravi       502     800
2          102   Ravi       502     800
3          103  Meena       503     600
4          104   John       504     700
5          104   John       505     700


## 1. What are Duplicate Records?

### Definition

Duplicate records are rows that repeat information already stored in a dataset. Sometimes every column is identical, while other times only important fields match. Duplicates may come from repeated uploads, manual entry, or combining files. Before removing them, we must check whether they represent the same real-world event.

### Business Example

A store uploads its daily sales file twice by mistake. The same order appears two times and makes total revenue look higher. The team checks order IDs to distinguish repeated uploads from real purchases. Only the accidental copies should be removed.

### AI/ML Use

Duplicate examples can make a machine learning model learn some observations more often than others. They may distort class frequencies and summary statistics. Repeated customer records can also appear across training and test sets, creating misleading evaluation results. Checking duplicates improves the reliability of the data.

### Python Example

In [3]:
print("Repeated rows:", df.duplicated().sum())

Repeated rows: 1


### Explanation

The duplicated() method compares each row with earlier rows. It marks a repeated row as True and the first occurrence as False. sum() counts the True values. This is a first check, not an automatic instruction to delete records.

## 2. Exact Duplicates

### Definition

Exact duplicates have identical values in every column. They often result from accidental copying or importing the same record twice. Pandas can find them by comparing complete rows. A repeated row may still require checking before deletion if identical-looking events are possible.

### Business Example

A billing system imports order 502 twice with the same customer, amount, and order number. The second row is an exact copy of the first. Counting both would overstate sales. The business can safely remove one after confirming that the order ID should be unique.

### AI/ML Use

Exact duplicates can overweight repeated examples during training. They may affect class balance and learned patterns. Removing confirmed accidental copies can make evaluation more realistic. However, validation against unique business keys is important.

### Python Example

In [5]:
print(df[df.duplicated()])

   Customer_ID  Name  Order_ID  Amount
2          102  Ravi       502     800


### Explanation

The expression df.duplicated() creates a True/False mask for repeated full rows. Using that mask inside df[...] displays only those rows. In this dataset, the second copy of order 502 is identified. The first copy remains unmarked by default.

## 3. Partial Duplicates

### Definition

Partial duplicates match on some columns but differ in others. They are common when a customer makes several purchases or a profile changes over time. Matching names or customer IDs does not prove that whole rows are errors. We must choose fields that define a unique event.

### Business Example

Customer 104 places orders 504 and 505 for the same amount. Their customer ID and name are repeated, but the order IDs differ. Both purchases may be genuine. Deleting one simply because the customer ID repeats would lose valid revenue.

### AI/ML Use

Partial duplicates matter when building customer-level or transaction-level models. Several rows per customer may be expected in a purchase prediction dataset. Treating them as errors can remove useful behavior history. The model’s unit of analysis determines whether repetition is valid.

### Python Example

In [7]:
print(df[df.duplicated(subset=["Customer_ID"], keep=False)])

   Customer_ID  Name  Order_ID  Amount
1          102  Ravi       502     800
2          102  Ravi       502     800
4          104  John       504     700
5          104  John       505     700


### Explanation

The subset argument checks only Customer_ID. keep=False marks every row belonging to a repeated customer ID. This includes valid repeat customers and accidental copies. Therefore, this output needs business interpretation rather than immediate deletion.

## 4. Identifying Duplicates

### Definition

Identifying duplicates means locating records that may refer to the same entity or event. You can compare complete rows or selected identifying columns. Duplicate counts show how common repetition is in a dataset. Inspecting the matching records helps decide which duplicates are mistakes.

### Business Example

An online marketplace reviews repeated orders before calculating monthly revenue. It first checks full-row duplicates and then repeated order IDs. Some repeated customers have legitimate multiple orders. The order ID helps identify suspicious records more accurately.

### AI/ML Use

Duplicate checks are part of data quality assessment before model training. They help prevent biased counts and repeated observations. Reviewing repeated IDs can also uncover data integration problems. These checks should happen before splitting data when leakage is a concern.

### Python Example

In [9]:
print("Exact duplicates:", df.duplicated().sum())
print("Repeated order IDs:", df.duplicated(subset=["Order_ID"]).sum())

Exact duplicates: 1
Repeated order IDs: 1


### Explanation

The first line counts rows that match on all columns. The second counts repeated Order_ID values, even if other fields differ. Comparing both counts shows the difference between exact and key-based duplicates. A repeated order ID should be reviewed using the business rule for order uniqueness.

## 5. duplicated()

### Definition

duplicated() is a Pandas method that marks repeated rows with Boolean values. By default, it checks all columns and keeps the first occurrence unmarked. The subset parameter lets you check only chosen columns. The keep parameter controls whether the first, last, or all matching rows are marked.

### Business Example

A customer database may contain repeated email addresses. The team uses duplicated(subset=["Email"]) to find possible repeat accounts. Some emails may belong to shared family accounts. The results are reviewed before accounts are merged.

### AI/ML Use

This method helps find repeated training examples or repeated identifiers. It is useful during exploratory data analysis and preprocessing. Choosing the right subset prevents accidental loss of valid rows. The output is a mask that can be used for inspection or filtering.

### Python Example

In [11]:
print(df.duplicated())
print(df.duplicated(subset=["Order_ID"], keep=False))

0    False
1    False
2     True
3    False
4    False
5    False
dtype: bool
0    False
1     True
2     True
3    False
4    False
5    False
dtype: bool


### Explanation

The first result marks repeated full rows after their first appearance. The second checks order IDs and marks all rows with a repeated order ID. True means the row meets the selected duplicate condition. The method detects repetition but does not delete anything.

## 6. drop_duplicates()

### Definition

drop_duplicates() removes rows that match the selected duplicate condition. By default, it checks all columns and keeps the first occurrence. You can specify subset and keep to change this behavior. Always inspect duplicates and confirm the business rule before applying it.

### Business Example

A sales team confirms that order 502 was imported twice. It removes the repeated copy while keeping the original order. Orders 504 and 505 remain because they are different transactions. The corrected dataset produces a more accurate revenue total.

### AI/ML Use

Removing confirmed duplicate observations can reduce bias in training data. It can also prevent repeated records from influencing metrics. However, dropping repeated customers indiscriminately can destroy useful transaction histories. The deduplication key must reflect the prediction task.

### Python Example

In [13]:
clean_df = df.drop_duplicates()
print(clean_df)

   Customer_ID   Name  Order_ID  Amount
0          101   Asha       501     500
1          102   Ravi       502     800
3          103  Meena       503     600
4          104   John       504     700
5          104   John       505     700


### Explanation

The method returns a new DataFrame without exact repeated rows. Here, the second copy of order 502 disappears. Both orders for customer 104 remain because their Order_ID values differ. The original df is unchanged unless you assign the result back to it.

## 7. Duplicate Detection Based on Selected Columns

### Definition

Selected-column duplicate detection checks repetition using only fields that matter for uniqueness. Common keys include Order_ID, Invoice_ID, or a combination of fields. A customer ID is not necessarily a unique transaction ID. Picking the wrong columns can incorrectly classify valid records as duplicates.

### Business Example

A courier company expects every shipment number to be unique. Two rows with the same shipment number need investigation, even if the delivery status differs. In contrast, one customer may have many shipment numbers. The company uses shipment number rather than customer name as its duplicate key.

### AI/ML Use

For ML datasets, selected columns help define whether observations are genuinely repeated. They are especially important after merging multiple data sources. Unique event IDs can reveal overlapping records. Careful key selection helps preserve meaningful repeated customer behavior.

### Python Example

In [15]:
print(df[df.duplicated(subset=["Order_ID"], keep=False)])

   Customer_ID  Name  Order_ID  Amount
1          102  Ravi       502     800
2          102  Ravi       502     800


### Explanation

The subset parameter tells Pandas to compare only Order_ID. keep=False displays both the original and repeated rows for an order. In this dataset, both copies of order 502 are shown. This makes it easier to review the suspected duplication.

## 8. Handling Duplicate Records

### Definition

Handling duplicates involves detecting, reviewing, and deciding what to keep. Some duplicates should be removed, while others should be updated or combined. The right action depends on identifiers, timestamps, and business rules. It is useful to keep a copy of the original data before changing it.

### Business Example

A customer profile appears twice with different phone numbers. Deleting one row might lose the newer contact details. The business checks update timestamps and combines correct information. In a sales table, separate orders from the same customer are retained.

### AI/ML Use

A consistent duplicate-handling process makes training datasets more trustworthy. Sometimes repeated events are informative rather than erroneous. Removing only verified mistakes protects important features and patterns. Documenting the decision also makes the ML pipeline reproducible.

### Python Example

In [17]:
reviewed_df = df.drop_duplicates(subset=["Order_ID"], keep="first")
print(reviewed_df[["Customer_ID", "Order_ID", "Amount"]])

   Customer_ID  Order_ID  Amount
0          101       501     500
1          102       502     800
3          103       503     600
4          104       504     700
5          104       505     700


### Explanation

This example assumes that each Order_ID identifies exactly one order. It keeps the first row for each order ID and removes later repetitions. Different orders from the same customer remain in the dataset. In a real system, check whether the first row is the correct one before choosing keep="first".

## 9. Business Rules for Duplicate Removal

### Definition

Business rules define what makes a record unique and how conflicts are resolved. For transactions, a unique order ID may be the correct key. For customer profiles, an account ID and update timestamp may matter more. Blind deletion is dangerous because repeated-looking rows can represent separate valid events.

### Business Example

A customer buys two ₹700 items on different orders. Both transactions should count toward revenue, even if customer and amount match. Removing one based only on those fields would undercount sales. The store removes records only when the same unique order ID was accidentally recorded twice.

### AI/ML Use

Incorrect duplicate removal can delete rare but useful examples from a model’s training data. It can also distort purchase frequency, spending totals, and class balance. Using documented rules makes preprocessing repeatable and explainable. Always verify whether repeated records are errors or legitimate observations.

### Python Example

In [19]:
print("Unique orders:", df["Order_ID"].nunique())
print("Rows after removing repeated order IDs:", len(df.drop_duplicates(subset=["Order_ID"])))

Unique orders: 5
Rows after removing repeated order IDs: 5


### Explanation

nunique() counts distinct order IDs. The second line counts rows after applying the order-ID uniqueness rule. These counts match when every remaining row represents one distinct order. This rule is suitable only if the business confirms that order IDs must be unique.